In [ ]:
pip install pyspark

In [ ]:
%%writefile students.csv
student_id,student_name
1,Arun
2,Priya
3,Kumar
4,Meena

In [ ]:
%%writefile courses.csv
course_id,course_name
101,Python
102,SQL
103,Power BI

In [ ]:
%%writefile enrollments.csv
enrollment_id,student_id,course_id,status
1,1,101,Completed
2,2,102,Enrolled
3,3,101,Completed
4,4,103,Dropped

In [ ]:
%%writefile progress.csv
enrollment_id,completion
1,100
2,45
3,100
4,20

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, when

spark = SparkSession.builder \
    .appName("CourseAnalysis") \
    .master("local[*]") \
    .getOrCreate()
students = spark.read.csv(
    "students.csv", header=True, inferSchema=True
)
courses = spark.read.csv(
    "courses.csv", header=True, inferSchema=True
)
enrollments = spark.read.csv(
    "enrollments.csv", header=True, inferSchema=True
)
progress = spark.read.csv(
    "progress.csv", header=True, inferSchema=True
)
data = enrollments.join(
    students, "student_id"
).join(
    courses, "course_id"
).join(
    progress, "enrollment_id"
)

print("Complete enrollment data:")
data.show()
summary = data.groupBy(
    "course_id", "course_name"
).agg(
    count("*").alias("total_enrolled"),
    count(when(col("status") == "Completed", 1))
        .alias("completed_students"),
    count(when(col("status") == "Dropped", 1))
        .alias("dropped_students"),
    avg("completion").alias("average_completion")
)

print("Course-wise summary:")
summary.show()
summary.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv("course_summary_output")

print("Course analysis completed!")
spark.stop()